In [21]:
# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = True
DOCKER_IMAGE_TAG = 'urbanopt-cli:1.2-ubuntu22'
# ─────────────────────────────────────────────────────────────────────────────

from functools import partial
from pathlib import Path

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / 'esbe_2026'
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / 'data' / 'templates'

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# update weather flag
update_weather_location = False
new_weather_epw = 'USA_FL_MacDill.AFB.747880_TMY3'
new_weather_climate_zone = '1A'


Working directory: /Users/nlong/working/urban-analysis/esbe
template_data_dir: /Users/nlong/working/urban-analysis/esbe/data/templates
analysis_dir: /Users/nlong/working/urban-analysis/esbe/esbe_2026


### Activity 0: Did everything install correctly?

In [3]:
baseline_analysis_dir = analysis_dir / 'activity_0'
if not baseline_analysis_dir.exists():
    baseline_analysis_dir.mkdir()
    
uo_coincident = UO(baseline_analysis_dir, 'coincident', template_dir=template_data_dir)
uo_diverse = UO(baseline_analysis_dir, 'diverse', template_dir=template_data_dir)

# Just print out one to make sure it looks right.
uo_coincident.info()

Template path: /Users/nlong/working/urban-analysis/esbe/data/templates
Working dir: /Users/nlong/working/urban-analysis/esbe/esbe_2026/activity_0
UO project: coincident
Log file: /Users/nlong/working/urban-analysis/esbe/esbe_2026/activity_0/coincident.log

URBANopt CLI version: 1.2.0

Usage:
  uo [options] [<command> [suboptions]]
 
Options:
  -v, --version    Print version and exit
  -h, --help       Show this help message

Commands:
  create         Make new things - project directory or files
  install_python Install python and other dependencies to run OpenDSS, DISCO,
GMT analysis
  update         Update files in an existing URBANopt project
  run            Use files in your directory to simulate district energy use
  process        Post-process URBANopt simulations for additional insights
  visualize      Visualize and compare results for features and scenarios
  validate       Validate results with custom rules
  opendss        Run OpenDSS simulation
  disco          Run DISCO a

### Activity 1: Example projects

In [4]:
activity_1_analysis_dir = analysis_dir / 'activity_1'
if not activity_1_analysis_dir.exists():
    activity_1_analysis_dir.mkdir()

uo_coincident = UO(activity_1_analysis_dir, 'coincident', template_dir=template_data_dir)
uo_diverse = UO(activity_1_analysis_dir, 'diverse', template_dir=template_data_dir)

uo_coincident.create_example_coincident_project()
uo_diverse.create_example_diverse_project()

# run sims faster
uo_coincident.set_number_parallel(8)
uo_diverse.set_number_parallel(8)

# copy over the weather files
uo_coincident.copy_over_weather()
uo_diverse.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(new_weather_epw, new_weather_climate_zone)
    uo_diverse.replace_weather_file_in_feature_and_mapper_file(new_weather_epw, new_weather_climate_zone)

    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')
    # uo_diverse.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

uo_coincident.create_scenarios('class_project_coincident.json')
uo_diverse.create_scenarios('class_project_diverse.json')

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name. 
uo_coincident.fix_dependencies_20260420('base_workflow.osw')
# TODO: fix wknd_op_hrs_start_time in the osw file as well.
uo_diverse.fix_dependencies_20260420('base_workflow.osw')



Creating a new project folder...
Weather data is provided for the example FeatureFile. Additional weather data files may be downloaded from energyplus.net/weather for free
If you use additional weather files, ensure they are added to the 'weather' directory. You will need to configure your mapper file and your osw file to use the desired weather file
We recommend using absolute paths for all commands, for cleaner output


Creating a new project folder...
Weather data is provided for the example FeatureFile. Additional weather data files may be downloaded from energyplus.net/weather for free
If you use additional weather files, ensure they are added to the 'weather' directory. You will need to configure your mapper file and your osw file to use the desired weather file
We recommend using absolute paths for all commands, for cleaner output

copying over weather files
copying over weather files

Building sample ScenarioFiles, assigning mapper classes to each feature from class_project_co

In [5]:
# Run the diverse baseline scenario
uo_coincident.run('class_project_coincident.json', 'baseline_scenario.csv')


Simulating features of 'class_project_coincident.json' as directed by 'baseline_scenario.csv'...

Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Using runner options from runner.conf file
Initializing runner with dirname: '/workspace/esbe_2026/activity_1/coincident' and options: {:file_version=>"0.1.0", :max_datapoints=>1000000000, :num_parallel=>8, :run_simulations=>true, :verbose=>false, :gemfile_path=>"/usr/local/urbanopt-cli-1.2.0/gems/Gemfile", :bundle_install_path=>"/usr/local/urbanopt-cli-1.2.0/gems"}
@bundle_without_string = ''
Gemfile.lock exists
needs_config = true
needs_platform = false
needs_update = true
run_dir '/workspace/esbe_2026/activity_1/coincident/run/baseline_sc

In [6]:
# Run the diverse baseline scenario
uo_diverse.run('class_project_diverse.json', 'baseline_scenario.csv')


Simulating features of 'class_project_diverse.json' as directed by 'baseline_scenario.csv'...

Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Using runner options from runner.conf file
Initializing runner with dirname: '/workspace/esbe_2026/activity_1/diverse' and options: {:file_version=>"0.1.0", :max_datapoints=>1000000000, :num_parallel=>8, :run_simulations=>true, :verbose=>false, :gemfile_path=>"/usr/local/urbanopt-cli-1.2.0/gems/Gemfile", :bundle_install_path=>"/usr/local/urbanopt-cli-1.2.0/gems"}
@bundle_without_string = ''
Gemfile.lock exists
needs_config = true
needs_platform = false
needs_update = true
run_dir '/workspace/esbe_2026/activity_1/diverse/run/baseline_scenario/1/

In [8]:
# post process the scenarios for both projects
uo_coincident.process_scenario('class_project_coincident.json', 'baseline_scenario.csv')
# uo_diverse.process_scenario('class_project_diverse.json', 'baseline_scenario.csv')

Post-processing URBANopt results
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems

Done



In [9]:
# visualize both the projects
uo_coincident.visualize_feature('class_project_coincident.json')
# uo_diverse.visualize_feature('class_project_diverse.json')


Creating visualizations for all Scenario results

Done



### Activity 2 EEMs


In [10]:
# This is the same as above, but in a new directory
activity_2_analysis_dir = analysis_dir / 'activity_2'
if not activity_2_analysis_dir.exists():
    activity_2_analysis_dir.mkdir()

uo_coincident = UO(activity_2_analysis_dir, 'coincident', template_dir=template_data_dir)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(8)

# copy over the weather files
uo_coincident.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(new_weather_epw, new_weather_climate_zone)
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')


# Create the scenarios
uo_coincident.create_scenarios('class_project_coincident.json')

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name. 
uo_coincident.fix_dependencies_20260420('base_workflow.osw')



Creating a new project folder...
Weather data is provided for the example FeatureFile. Additional weather data files may be downloaded from energyplus.net/weather for free
If you use additional weather files, ensure they are added to the 'weather' directory. You will need to configure your mapper file and your osw file to use the desired weather file
We recommend using absolute paths for all commands, for cleaner output

copying over weather files

Building sample ScenarioFiles, assigning mapper classes to each feature from class_project_coincident.json

Done



In [ ]:
# The need to create a new mapper (per the instructions) is not needed, since there is
# a classproject_scenario.csv, just run that one.

# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run('class_project_coincident.json', 'baseline_scenario.csv')

In [13]:
# Manually enable some of the measures in the mappers/ClassProject.rb file (set skip to false)
uo_coincident.run('class_project_coincident.json', 'classproject_scenario.csv')



Simulating features of 'class_project_coincident.json' as directed by 'classproject_scenario.csv'...

Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Using runner options from runner.conf file
Initializing runner with dirname: '/workspace/esbe_2026/activity_2/coincident' and options: {:file_version=>"0.1.0", :max_datapoints=>1000000000, :num_parallel=>8, :run_simulations=>true, :verbose=>false, :gemfile_path=>"/usr/local/urbanopt-cli-1.2.0/gems/Gemfile", :bundle_install_path=>"/usr/local/urbanopt-cli-1.2.0/gems"}
@bundle_without_string = ''
Gemfile.lock exists
needs_config = true
needs_platform = false
needs_update = true
run_dir '/workspace/esbe_2026/activity_2/coincident/run/classpr

In [14]:
# process the scenarios
uo_coincident.process_scenario('class_project_coincident.json', 'baseline_scenario.csv')
uo_coincident.process_scenario('class_project_coincident.json', 'classproject_scenario.csv')

# create the scenario and feature visualizations
uo_coincident.visualize_scenario('class_project_coincident.json', 'baseline_scenario.csv')
uo_coincident.visualize_scenario('class_project_coincident.json', 'classproject_scenario.csv')

# and then the feature visualization to compare
uo_coincident.visualize_feature('class_project_coincident.json')

Post-processing URBANopt results
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems

Done

Post-processing URBANopt results
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems

Done


Creating visualizations for all Scenario results

Done

Creating visualizations for Feature results in the Scenario

Done


Creating visualizations for all Scenario results

Done

Creating visualizations for Feature results in the Scenario

Done


Creating visualizations for all Scenario results

Done



### Activity 3: REopt

In [15]:
# This is the same as above, but in a new directory
activity_3_analysis_dir = analysis_dir / 'activity_3'
if not activity_3_analysis_dir.exists():
    activity_3_analysis_dir.mkdir()

uo_coincident = UO(activity_3_analysis_dir, 'coincident', template_dir=template_data_dir)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(12)

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(new_weather_epw, new_weather_climate_zone)
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

# Create the scenarios
uo_coincident.create_scenarios('class_project_coincident.json')

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name. 
uo_coincident.fix_dependencies_20260420('base_workflow.osw')



Creating a new project folder...
Weather data is provided for the example FeatureFile. Additional weather data files may be downloaded from energyplus.net/weather for free
If you use additional weather files, ensure they are added to the 'weather' directory. You will need to configure your mapper file and your osw file to use the desired weather file
We recommend using absolute paths for all commands, for cleaner output


Building sample ScenarioFiles, assigning mapper classes to each feature from class_project_coincident.json

Done



In [16]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run('class_project_coincident.json', 'baseline_scenario.csv')

# post process/visualize the baseline
uo_coincident.process_scenario('class_project_coincident.json', 'baseline_scenario.csv')
uo_coincident.visualize_scenario('class_project_coincident.json', 'baseline_scenario.csv')
uo_coincident.visualize_feature('class_project_coincident.json')



Simulating features of 'class_project_coincident.json' as directed by 'baseline_scenario.csv'...

Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Using runner options from runner.conf file
Initializing runner with dirname: '/workspace/esbe_2026/activity_3/coincident' and options: {:file_version=>"0.1.0", :max_datapoints=>1000000000, :num_parallel=>12, :run_simulations=>true, :verbose=>false, :gemfile_path=>"/usr/local/urbanopt-cli-1.2.0/gems/Gemfile", :bundle_install_path=>"/usr/local/urbanopt-cli-1.2.0/gems"}
@bundle_without_string = ''
Gemfile.lock exists
needs_config = true
needs_platform = false
needs_update = true
run_dir '/workspace/esbe_2026/activity_3/coincident/run/baseline_s

In [17]:
# Create the scenario mapper file that is enabled with the REopt assumptions
uo_coincident.create_reopt_scenario('class_project_coincident.json', 'baseline_scenario.csv')

# Confirm in the REopt_baseline_scenario file that the REopt assumptions are now enabled (look for the REopt Assumptions section) in the file


Creating ScenarioFile with REopt functionality, extending from coincident/baseline_scenario.csv...

Done



In [18]:
# Run the REopt baseline scenario (which only runs the baseline buildings). REopt
# will be postprocessed in the next step.
uo_coincident.run('class_project_coincident.json', 'REopt_baseline_scenario.csv')



Simulating features of 'class_project_coincident.json' as directed by 'REopt_baseline_scenario.csv'...

Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Using runner options from runner.conf file
Initializing runner with dirname: '/workspace/esbe_2026/activity_3/coincident' and options: {:file_version=>"0.1.0", :max_datapoints=>1000000000, :num_parallel=>12, :run_simulations=>true, :verbose=>false, :gemfile_path=>"/usr/local/urbanopt-cli-1.2.0/gems/Gemfile", :bundle_install_path=>"/usr/local/urbanopt-cli-1.2.0/gems"}
@bundle_without_string = ''
Gemfile.lock exists
needs_config = true
needs_platform = false
needs_update = true
run_dir '/workspace/esbe_2026/activity_3/coincident/run/reop

In [25]:
# if the REopt errors with cert issues, then look at this help site,
#   But where do you run the bundle update command?
#   https://docs.urbanopt.net/developer_resources/known_issues.html

# Also, try to access the reopt site directly to make sure the API is correct
#  https://developer.nrel.gov/api/reopt/v1/?API_KEY=ganRGlzka5XeOnae21cepxb1vkIX57fCsGc6x2EZ

uo_coincident.process_reopt_scenario('class_project_coincident.json', 'REopt_baseline_scenario.csv', individual_features=False)
# uo_coincident.process_reopt_scenario('class_project_coincident.json', 'REopt_baseline_scenario.csv', individual_features=True)

Post-processing URBANopt results
Defaulting to .bundle/install (and ignoring system wide: Bundler.configured_bundle_path.base_path=/usr/local/urbanopt-cli-1.2.0/gems/ruby/3.2.0)
Bundle final path is set to: /usr/local/urbanopt-cli-1.2.0/gems
Using default scenario assumptions file: /workspace/esbe_2026/activity_3/coincident/reopt/multiPV_assumptions.json

Running the REopt post-processor with scenario assumptions file: /workspace/esbe_2026/activity_3/coincident/reopt/multiPV_assumptions.json

Post-processing entire scenario with REopt

Done



In [26]:
# create the scenario and feature visualizations
uo_coincident.visualize_scenario('class_project_coincident.json', 'baseline_scenario.csv')
uo_coincident.visualize_scenario('class_project_coincident.json', 'REopt_baseline_scenario.csv')

# and then the feature visualization to compare
uo_coincident.visualize_feature('class_project_coincident.json')


Creating visualizations for all Scenario results

Done

Creating visualizations for Feature results in the Scenario

Done


Creating visualizations for all Scenario results

Done

Creating visualizations for Feature results in the Scenario

Done


Creating visualizations for all Scenario results

Done



### Gather the data for the OSA / PAT files


In [ ]:
# read in the feature file
import json
import shutil

# get the results from the activity 3 folder
uo_coincident = UO(analysis_dir / 'activity_3', 'coincident', template_dir=template_data_dir)
activity_pat_analysis_dir = analysis_dir / 'activity_2_pat'
if not activity_pat_analysis_dir.exists():
    activity_pat_analysis_dir.mkdir()

feature_data = uo_coincident.project_path / 'class_project_coincident.json'
run_path = uo_coincident.project_path / 'run' / 'baseline_scenario' 

files_to_copy = []

feature_json = None

with open(feature_data, 'r') as f:
    feature_json = json.load(f)

    for feature in feature_json['features']:
        if feature['properties']['type'] == 'Building':
            feature_id = feature['properties']['id']
            feature_name = feature['properties']['name']

            # go to the run directory and grab the OSM file
            osm_file = run_path / feature_id / 'in.osm'
            new_filename = f'{feature_name}.osm'
            files_to_copy.append({ 
                'base_dir': 'seeds',
                'feature_name': feature_name,
                'file': osm_file,
                'new_filename': new_filename,
            })
    
    # copy all the files in the directory
    files = (uo_coincident.project_path / 'weather').glob('*')
    for file in files:
        files_to_copy.append({
            'base_dir': 'weather',
            'file': file,
            'new_filename': file.name,
        })
    
        
    
print(files_to_copy)
for base_dir in (file['base_dir'] for file in files_to_copy):
    if not (activity_pat_analysis_dir / base_dir).exists():
        (activity_pat_analysis_dir / base_dir).mkdir()
    
for file in files_to_copy:
    shutil.copy(file['file'], activity_pat_analysis_dir / file['base_dir'] / file['new_filename'])


# NOW FOR THE CRAZY PART...

# Run the uo_building_to_osa.rb file to convert the osa data. Then grab the measures from
# that result and place in the activity_2_pat folder

